In [1]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append('..')


import utilities.functions as functions

from utilities.functions import (
    #load_data,
    check_key_uniqueness,
    merge_df,
    load_orders,
   orders_month_cli
)
from utilities.load_data import (

   load_data,
   count_null

)

In [2]:
from pathlib import Path
BASE_PATH = Path("/Users/maceli/ifood_cs/dados") 



Load the data --- se necessario salvar em stage - neste momento o estara comentado

In [ ]:
URL_CONSUMER = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/consumer.csv.gz"
df_consumer = load_data(URL_CONSUMER)[["customer_id", "active","created_at"]]

In [ ]:
URL_RESTAURANT ="https://data-architect-test-source.s3-sa-east-1.amazonaws.com/restaurant.csv.gz"
df_restaurant= load_data(URL_RESTAURANT)

In [ ]:
ab_test_url = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/ab_test_ref.tar.gz"
df_ab = load_data(ab_test_url)

In [ ]:
#save_parquet(df_consumer, "stage", "df_consumer.parquet")
#save_parquet(df_restaurant, "stage", "df_restaurant.parquet")
#save_parquet(df_consumer, "stage", "df_consumer.parquet")

Bronze layer - verify duplicates e nulo. e se necessario remover

In [ ]:
check_key_uniqueness(df_consumer, ["customer_id","active","created_at"])

In [ ]:
check_key_uniqueness(df_restaurant, ["id"])

In [ ]:
check_key_uniqueness(df_ab, ["customer_id","is_target"])
print(df_ab[df_ab["customer_id"].isna()])


Valida nullos

In [ ]:
df_consumer_null=count_null(df_consumer)
df_consumer_null.head()

In [ ]:
df_restaurant_null=count_null(df_restaurant)
df_restaurant_null.head()

In [ ]:
df_ab_null=count_null(df_ab)
df_ab_null.head()

In [ ]:
df_ab_np = df_ab[~df_ab["customer_id"].isna()]

Antes de carregar a base ordens sera definido o publico todal, e da base de ordem serao filtrados somentes os clientes elegiceis

In [ ]:
df_publico=merge_df(df_ab_np,df_consumer,['customer_id'],'left')

In [ ]:
clientes_mes = (
    df_publico.groupby(['active', 'is_target'], dropna=False)
              ['customer_id']
              .nunique()
              .reset_index(name='numero_clientes_distintos')
)
clientes_mes

In [ ]:
df_publico = df_publico.dropna(subset=['active'])

In [ ]:
clientes_mes = (
    df_publico.groupby(['active', 'is_target'], dropna=False)
              ['customer_id']
              .nunique()
              .reset_index(name='numero_clientes_distintos')
)
clientes_mes

Publico definido e todos os clientes marcados no teste a/b e existentes na base de clientes

Da base de ordens serao filtrados todos os clientes com orden nos meses de de dezembro e janeiro

In [ ]:
df_ab=df_ab.head(100)

In [ ]:
URL_ORDERS = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/order.json.gz"


COLUMNS_TO_DROP = [
    'cpf','customer_name','delivery_address_city','delivery_address_country',
    'delivery_address_district','delivery_address_external_id',
    'delivery_address_latitude','delivery_address_longitude',

    'delivery_address_zip_code','items',
    'merchant_latitude','merchant_longitude','merchant_timezone',
    'order_scheduled','order_scheduled_date'
]

customer_ids = df_ab["customer_id"].astype(str).unique()

df_orders = load_orders(
    url=URL_ORDERS,
    customer_ids=customer_ids,
    columns_to_drop=COLUMNS_TO_DROP
)

df_orders.head()


In [ ]:
print(check_key_uniqueness(df_orders, ["customer_id","merchant_id","order_id"]))
print(check_key_uniqueness(df_orders, ["customer_id","merchant_id","order_id","order_created_at"]))

In [ ]:
df_orders_null=count_null(df_orders)
df_orders_null.head()

Add orders para a base de publico

In [ ]:
df_publico_orders=merge_df(df_publico,df_orders,['customer_id'],'inner')

In [ ]:
#df_publico_orders.to_parquet(BASE_PATH / "silver" / "df_publico_orders.parquet", index=False)

Construcao de chave unica, e sumarizacoes visao cliente

In [4]:
df_cliente,df_publico=orders_month_cli(df_publico_orders)

In [6]:
df_cliente.head()

,customer_id,is_target,order_created_month,num_pedidos_mes,num_pedidos_hist,total_amount_mes,ticket_medio
0,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,1,17,19,206.50,12.147059
1,755e1fa18f25caec5edffb188b13fd844b2af8cf5adedc...,target,12,2,19,22.50,11.250000
2,b821aa8372b8e5b82cdc283742757df8c45eecdd72adf4...,control,1,5,6,267.88,53.576000
3,b821aa8372b8e5b82cdc283742757df8c45eecdd72adf4...,control,12,1,6,20.00,20.000000
4,d425d6ee4c9d4e211b71da8fc60bf6c5336b2ea9af9cc0...,control,1,20,31,1300.89,65.044500


In [7]:
df_stats_hist = df_cliente.groupby(['num_pedidos_hist']).agg(
    total_clientes=('customer_id', 'nunique')  
).round(2)

df_stats_hist['pct_total'] = (df_stats_hist['total_clientes'] / df_stats_hist['total_clientes'].sum() * 100).round(2)

df_stats_hist.head(20)

,total_clientes,pct_total
num_pedidos_hist,,
1,8,8.0
2,12,12.0
3,4,4.0
4,7,7.0
5,7,7.0
6,9,9.0
7,5,5.0
8,2,2.0
9,4,4.0


In [ ]:
#df_publico_orders = pd.read_parquet(BASE_PATH / "silver" / "df_publico_orders.parquet")

Uma linha por cliente, com as variaveis necessarias

Salvar base visao cliente em gold layer

In [10]:
df_cliente.to_parquet(BASE_PATH / "gold" / "df_clientes.parquet", index=False)

In [9]:
df_publico.to_parquet(BASE_PATH / "gold" / "df_publico.parquet", index=False)

In [ ]:
df_stats_mes = df_pub_un.groupby(['pedidos_sum', 'order_created_month']).agg(
    total_clientes=('customer_id', 'nunique')
)

df_stats_mes['pct_total_mes'] = (
    df_stats_mes['total_clientes'] /
    df_stats_mes.groupby('order_created_month')['total_clientes'].transform('sum') * 100
).round(2)

df_stats_mes.head(20)
